# Demo: Renormalization Group Evolution (RGE)

This notebook demonstrates how to use `RGEsolver` to study the scale dependence of dark sector couplings and the scalar VEV in the dark $U(1)$ model.

**Pipeline step:** RGE is the first layer — all other modules (Veff, bounce solver, FOPT) call it internally to evaluate couplings at a given renormalization scale $\mu$.

**Modules used:** `src/RGE/RGEsolver.py`, `src/RGE/VeffFunc_RGE.py`

**Contents:**
1. Running of dark sector couplings $g_D^2$, $\lambda_S$, $m_S^2$ with $\mu$
2. Scale dependence of the scalar VEV $\langle\phi\rangle(\mu)$
3. Coleman–Weinberg scale compensation: $v_{\rm CW}(\mu)/\mu \approx e^{1/6}/g_D(\mu)$

In [ ]:
%load_ext autoreload
%autoreload 2

%run ../startup.py

In [ ]:
# Instantiate the RGE solver and the effective potential
rge_solver = rge.RGESolver()
veff_obj   = veff.VeffRGE(vt_table_path="../VT_integralNumeric.dat")
solver_ht  = bs_ht.BounceSolverHighT(veff=veff_obj)

## 1. Running of dark sector couplings

Solve the one-loop RGE system from $\mu_0 = 1$ across 14 decades. The beta functions are defined in `RGEsolver.py`:
$$\beta_{g_D^2} = \frac{g_D^4}{24\pi^2}, \qquad \beta_{\lambda_S} = \frac{3g_D^4 - 6g_D^2\lambda_S + 10\lambda_S^2}{8\pi^2}, \qquad \beta_{m_S^2} = -m_S^2\frac{3g_D^2 - 4\lambda_S}{8\pi^2}$$

In [ ]:
# === Initial conditions at mu0 = 1 ===
gD0       = 0.6
lambdaS0  = 0.0
ms0       = 1e-10
logT_span = (0, 10)   # log(mu) range: mu in [1, e^10]

# Integrate RGE once with dense output for smooth curves
sol = solve_ivp(
    rge_solver.RGEs_logMu,
    logT_span,
    [gD0**2, lambdaS0, ms0],
    dense_output=True,
    max_step=0.1
)

# Evaluate at 200 log-spaced scale values
mu_vals   = np.logspace(-10, 4, 200)
logmu_vals = np.log(mu_vals)
y_vals    = sol.sol(logmu_vals)          # shape: (3, 200)
gD2_vals, lambdaS_vals, mS_vals = y_vals

In [ ]:
fig, ax = new_figure(figsize=(7, 5), dpi=120)
fig.patch.set_facecolor('white')

ax.plot(mu_vals, gD2_vals / (2 * np.pi), lw=1.8, color='#845ec2', label=r'$g_D^2 / 2\pi$')
ax.plot(mu_vals, lambdaS_vals,            lw=1.8, color='#ff6f91', label=r'$\lambda_S$')
ax.plot(mu_vals, mS_vals,                 lw=1.8, color='#845ec2', ls='--', label=r'$m_S^2$')

ps.apply_standard_formatting(
    ax,
    xlog=True, ylog=False,
    xlabel=r'$\mu/\mu_0$',
    ylabel=r'Running parameter',
    ylim=(-0.18, 0.1),
)
ax.axhline(0, color='k', lw=0.8, ls=':')
ax.legend(loc='lower right', frameon=False, fontsize=14)
plt.tight_layout()

## 2. Scale dependence of the VEV

For each value of the initial coupling $g_D(\mu_0)$, run the RGE to scale $\mu$ and minimize $V_{\rm eff}^{\rm T=0}(\phi;\mu)$ to find the VEV $\langle\phi\rangle(\mu)$. The physical VEV must be $\mu$-independent; any residual dependence indicates higher-order corrections.

In [ ]:
mu0      = 1.0
ls0, mS0 = 0.0, 0.0
gD0_list = [0.5, 0.6, 0.7, 0.8]
mu_sweep = np.logspace(-3, 3, 200)

results = {}
for gD0 in gD0_list:
    v_over_mu0 = []
    for mu in mu_sweep:
        gD_mu, _, _ = rge_solver.run_params(mu=mu, gD0=gD0, lambdaS0=ls0, mS0=mS0)
        v_mu = solver_ht.phi_min_Veff0(T=1, gD=gD_mu, scale=mu, ls0=ls0)
        v_over_mu0.append(v_mu / mu0)
    results[gD0] = np.array(v_over_mu0)

In [ ]:
colors = ['#845ec2', '#ff6f91', 'limegreen', 'dodgerblue']

fig, ax = new_figure(figsize=(7, 5), dpi=120)
fig.patch.set_facecolor('white')

for (gD0, vals), c in zip(results.items(), colors):
    ax.plot(mu_sweep / mu0, vals, color=c, lw=1.8, label=fr'$g_D(\mu_0)={gD0}$')
    # Analytic asymptotic value at mu = mu0: v_CW = exp(1/6) * mu0 / gD0
    ax.axhline(np.exp(1/6) / gD0, color=c, ls='--', lw=1.0, alpha=0.7)

ps.apply_standard_formatting(
    ax,
    xlog=True, ylog=False,
    xlabel=r'$\mu/\mu_0$',
    ylabel=r'$\langle\phi\rangle / \mu_0$',
    ylim=(0.9, 4),
    xlim=(3e-3, 1e2),
)
ax.legend(loc='best', frameon=False, fontsize=13, ncol=2)
ax.text(0.97, 0.05, 'dashed = analytic CW limit', transform=ax.transAxes,
        ha='right', fontsize=10, color='gray')
plt.tight_layout()

## 3. Coleman–Weinberg scale compensation

At one loop, the CW minimum satisfies $v_{\rm CW}(\mu) = e^{1/6}\,\mu / g_D(\mu)$. Plotting $v_{\rm CW}(\mu)/\mu = e^{1/6}/g_D(\mu)$ shows how the slow running of $g_D$ keeps the VEV approximately flat — the remaining slope is the residual scheme dependence.

In [ ]:
fig, ax = new_figure(figsize=(7, 5), dpi=120)
fig.patch.set_facecolor('white')

for gD0, c in zip([0.5, 0.6, 0.7, 0.8], colors):
    ratio = []
    for mu in mu_sweep:
        gD_mu, _, _ = rge_solver.run_params(mu=mu, gD0=gD0, lambdaS0=0.0, mS0=0.0)
        ratio.append(np.exp(1/6) / gD_mu if gD_mu > 0 else np.nan)
    ax.plot(mu_sweep / mu0, ratio, color=c, lw=1.8, label=fr'$g_D(\mu_0)={gD0}$')

ps.apply_standard_formatting(
    ax,
    xlog=True, ylog=True,
    xlabel=r'$\mu/\mu_0$',
    ylabel=r'$v_{\rm CW}(\mu)/\mu = e^{1/6}/g_D(\mu)$',
)
ax.legend(loc='best', frameon=False, fontsize=13)
plt.tight_layout()